# Network export and lineage communication

Explore inferred regulatory networks, reload models, and compare communication between two lineages. The synthetic demonstration uses paired models; saved models can be paired, unpaired or mixed.

The default `demo` mode prepares compact synthetic inputs and trains small demonstration models. Set `data_source="trained_models"` to use your own saved models and ligand-receptor resource.

Install the notebook dependencies with `pip install -e ".[full,tutorial]"` from a source checkout, select that Python kernel, and run the cells in order.


In [ ]:
%matplotlib inline
from pathlib import Path
import os
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from draft_paired.utils import load_yaml_config

torch.set_num_threads(2)
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/draft_paired").is_dir()), Path.cwd())
os.chdir(project_root)
workspace = Path(os.environ.get("DRAFT_TUTORIAL_OUTPUT_DIR", project_root / "result/downstream_example")).resolve()
data_source = "demo"  # Choose demo or trained_models.
retrain = data_source == "demo"
saved_configs = [Path("prepared/lineage1/train.yaml"), Path("prepared/lineage2/train.yaml")]
ligand_receptor_resource = Path("data/ligand_receptor.csv")
early_stage_counts = [1, 1]  # Choose the biological early windows for your models.


## Select model inputs

Demo mode prepares simulated paired RNA/ATAC, labels, trees and candidate priors under `workspace/lineage1/inputs/` and `workspace/lineage2/inputs/`. It writes `graphs.pt` and `train.yaml` in each lineage's `prepared/` directory.

For your trained models, set `data_source="trained_models"` and give their training-config paths in `saved_configs` above. Use the saved `run_config.yaml` when training selected a custom output directory.


In [ ]:
from draft_paired.tutorial_data import prepare_demo_data
from draft_paired.workflow import preprocess_lineage

if data_source == "demo":
    raw_paths = [
        prepare_demo_data(workspace / "lineage1" / "inputs", mode="paired", lineage="lineage1"),
        prepare_demo_data(workspace / "lineage2" / "inputs", mode="paired", lineage="lineage2", seed=102),
    ]
    config_paths = []
    for i, raw_path in enumerate(raw_paths, 1):
        prepared = workspace / f"lineage{i}" / "prepared"
        if not ((prepared / "graphs.pt").is_file() and (prepared / "train.yaml").is_file()):
            preprocess_lineage(raw_path, "paired", prepared)
        config_paths.append(prepared / "train.yaml")
elif data_source == "trained_models":
    config_paths = [path.resolve() for path in saved_configs]
    for path in config_paths:
        if not path.is_file():
            raise FileNotFoundError(f"Set saved_configs to existing training configs: {path}")
else:
    raise ValueError("Choose demo or trained_models explicitly.")

configs = [load_yaml_config(path) for path in config_paths]
display(pd.DataFrame({"lineage": [c["lineage"] for c in configs], "config": list(map(str, config_paths))}))


## Load models and export networks

Demo mode fits the small models first. Own-model mode reuses the saved checkpoints. Embeddings and networks are exported separately for each lineage.


In [ ]:
import importlib
from draft_paired.workflow import train_graph_model

reports = []
for path, cfg in zip(config_paths, configs):
    if retrain:
        train_graph_model(path)
    mode = cfg.get("input_mode", "paired")
    package = f"draft_{mode}"
    importlib.import_module(f"{package}.reproduce").reproduce_lineage_model(path)
    reports.append(importlib.import_module(f"{package}.analysis.network").export_lineage_networks(path))


## Inspect the inferred networks

In demo mode, each `workspace/lineage*/inputs/result/` directory contains `model/` checkpoints, `reproduce/` embeddings and smoothness summaries, and `network/` stage-specific TF-TG tables. For your models, output locations come from their configs. The next cell lists network sizes and previews one table. Network scores rank model-supported regulatory relations.


In [ ]:
stage_counts = []
for cfg in configs:
    files = sorted(Path(cfg["network"]["output_dir"]).glob("t*_TF_TG.tsv"))
    for file in files:
        stage_counts.append({"lineage": cfg["lineage"], "stage": file.stem.split("_")[0], "edges": len(pd.read_csv(file, sep="\t"))})
display(pd.DataFrame(stage_counts))
first_network = sorted(Path(configs[0]["network"]["output_dir"]).glob("t*_TF_TG.tsv"))[0]
display(pd.read_csv(first_network, sep="\t").head())
pd.DataFrame(stage_counts).pivot(index="stage", columns="lineage", values="edges").plot.bar()
plt.ylabel("Exported TF-TG edges")
plt.tight_layout()
plt.show()


## Compare communication between lineages

Demo mode creates a ligand-receptor table. For your own models, set `ligand_receptor_resource` to your local CSV and choose `early_stage_counts` in the setup cell. The CSV uses the [ligand-receptor format](input_formats.md#communication-between-two-lineages).

The program reads RNA and cell labels from the training configs, aligns regulatory-network stages with DTW, and calculates communication in both directions. Paired, unpaired and mixed comparisons share this interface. Results go to `workspace/communication/`, including `ccc_curves.tsv`, `ccc_pairs.tsv` and the DTW path. See the [command-line guide](downstream.md#draft-ccc).


In [ ]:
from draft_paired.tutorial_data import prepare_demo_ccc
from draft_paired.ccc_inputs import build_ccc_config
from draft_paired.analysis.ccc import run_ccc_analysis

if data_source == "demo":
    ccc_source = prepare_demo_ccc(config_paths, workspace / "communication")
else:
    ccc_source = build_ccc_config(config_paths, ligand_receptor_resource, early_stage_counts, output=workspace / "communication")

ccc_report = run_ccc_analysis(ccc_source)
display(pd.read_csv(ccc_report["pairs_path"], sep="\t"))


In [ ]:
curves = pd.read_csv(ccc_report["curves_path"], sep="\t")
for direction, part in curves[curves.method == "DRAFT"].groupby("direction"):
    part.pivot(index="stage_index", columns=["ligand", "receptor"], values="score").plot(title=direction)
    plt.ylabel("Communication score")
    plt.tight_layout()
    plt.show()


## Further analyses

Continue with [edge/topic clustering and TF regulons](downstream.md#draft-edge-cluster-and-draft-topic). The [input guide](input_formats.md) describes preparing graphs for a new dataset.
